[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-6-5-capstone-full-stack-project.ipynb)

# Lesson 6.5 · Capstone: Full-Stack Project

**Free Python For Development** · Topic 6: Advanced Python & Real-World Projects

_FastAPI + MongoDB with auth, CI/CD._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-6-5)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q fastapi httpx httpx2 "python-jose[cryptography]" passlib "bcrypt==4.0.1" mongomock-motor

## Big Picture & FastAPI Backend

You have spent weeks mastering Python — async programming, web scraping, automation, packaging and deployment. Now it is time to put everything together in a **capstone project** that mirrors real-world development.

A typical full-stack Python application lets users register, log in and manage their own data — for example a productivity tracker or task manager. The backend is built with **FastAPI**: a modern, high-performance framework that embraces Python’s async capabilities so the service can handle many concurrent requests smoothly.

> 💡 **Key insight:** FastAPI gives you automatic OpenAPI docs, request validation via Pydantic, and native `async`/`await` support — the same foundation used by many production APIs.

### Minimal FastAPI Skeleton

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Task Tracker API", version="1.0.0")

class TaskCreate(BaseModel):
    title: str
    done: bool = False

class Task(TaskCreate):
    id: str

# In-memory store for the demo
tasks: dict[str, Task] = {}

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/tasks", response_model=Task, status_code=201)
async def create_task(payload: TaskCreate):
    task_id = str(len(tasks) + 1)
    task = Task(id=task_id, **payload.model_dump())
    tasks[task_id] = task
    return task

@app.get("/tasks", response_model=list[Task])
async def list_tasks():
    return list(tasks.values())

**Try the API** — `TestClient` sends real HTTP requests to the app without starting a server:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
print(client.get("/health").json())
print(client.post("/tasks", json={"title": "Design the database"}).json())
print(client.get("/tasks").json())

**Why FastAPI fits the capstone**

- Async-ready for high concurrency.
- Pydantic models keep data contracts clear.
- Automatic interactive documentation.
- Easy to extend with auth, databases and middleware.

## MongoDB Storage & JWT Authentication

User data needs a durable home. **MongoDB** is a natural fit: it stores flexible JSON-like documents, so adding new fields (tags, notes, preferences) later does not require schema migrations.

**MongoDB strengths**

Document model matches Python dictionaries. Easy horizontal scaling. Great for evolving product requirements.

**Authentication goal**

Only authenticated users may read or write their own data. JWT tokens act as short-lived digital passports after login.

### Connecting the Pieces

In [ ]:
from datetime import datetime, timedelta, timezone
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, JWTError
# Colab has no MongoDB server: mongomock-motor is an in-memory stand-in with Motor's async API.
# With a real server use: from motor.motor_asyncio import AsyncIOMotorClient
from mongomock_motor import AsyncMongoMockClient
from passlib.context import CryptContext

SECRET_KEY = "change-me-in-production"
ALGORITHM = "HS256"
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

client = AsyncMongoMockClient()   # real server: AsyncIOMotorClient("mongodb://localhost:27017")
db = client["tasktracker"]

def create_access_token(data: dict, expires_minutes: int = 60):
    to_encode = data.copy()
    expire = datetime.now(timezone.utc) + timedelta(minutes=expires_minutes)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

async def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        email: str = payload.get("sub")
        if email is None:
            raise HTTPException(status_code=401, detail="Invalid token")
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid token")
    user = await db.users.find_one({"email": email})
    if not user:
        raise HTTPException(status_code=401, detail="User not found")
    return user

@app.post("/register")
async def register(email: str, password: str):
    if await db.users.find_one({"email": email}):
        raise HTTPException(status_code=400, detail="Email already registered")
    hashed = pwd_context.hash(password)
    await db.users.insert_one({"email": email, "hashed_password": hashed})
    return {"msg": "registered"}

@app.post("/token")
async def login(email: str, password: str):
    user = await db.users.find_one({"email": email})
    if not user or not pwd_context.verify(password, user["hashed_password"]):
        raise HTTPException(status_code=401, detail="Incorrect credentials")
    token = create_access_token({"sub": email})
    return {"access_token": token, "token_type": "bearer"}

**Try it** — add a protected endpoint that uses `get_current_user`, then register, log in and call it with the token:

In [ ]:
from fastapi.testclient import TestClient

@app.get("/me")
async def read_me(user: dict = Depends(get_current_user)):
    return {"email": user["email"]}

client = TestClient(app)
creds = {"email": "student@example.com", "password": "learn-python"}

print(client.post("/register", params=creds).json())
token = client.post("/token", params=creds).json()["access_token"]
print("token:", token[:30] + "…")
print(client.get("/me", headers={"Authorization": f"Bearer {token}"}).json())
print("no token →", client.get("/me").status_code)

**Security essentials**

- Hash passwords (never store plain text).
- Issue short-lived JWT access tokens.
- Protect every private route with `Depends(get_current_user)`.
- Keep secrets in environment variables, not source code.

## CI/CD & Assembling a Professional System

In professional teams, every change is automatically tested and, when green, deployed. That is the role of a **CI/CD pipeline**. After you push a new feature, the pipeline installs dependencies, runs tests, builds a container image and ships it to the server — without manual steps.

### Example GitHub Actions Workflow

This GitHub Actions pipeline tests every push and deploys `main`. It runs on GitHub, not in Colab:

```yaml
# .github/workflows/ci-cd.yml
name: CI/CD

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -r requirements.txt pytest
      - run: pytest --cov=app

  build-and-deploy:
    needs: test
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Build Docker image
        run: docker build -t my-task-api:${{ github.sha }} .
      - name: Push & deploy
        run: |
          # push to registry, then restart service
          echo "Deploying ${{ github.sha }}"
```

**Capstone checklist**

- FastAPI app with clear Pydantic models.
- MongoDB (or another DB) for persistent storage.
- JWT / OAuth2 authentication protecting private routes.
- Automated tests + CI pipeline.
- Containerised deployment (Docker + cloud or VPS).

**What you have built**

A secure, async API, a flexible document store, identity management, and an automated path from commit to production.

**What you can do next**

Add a frontend, rate limiting, observability, background workers, or multi-tenant features — the same architecture scales.

> 💡 **Big takeaway:** The capstone is not only about writing endpoints. It is about connecting FastAPI, MongoDB, authentication and CI/CD into one reliable system. With these skills you can design, secure and ship modern Python applications the same way professional teams do every day.

---

✅ **Lesson complete!**

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).